In [2]:
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, PG_ENGINE, BRONZE_DB

# ETL Raw → Bronze
Lee cada tabla de MySQL (Sakila) con pandas y la carga en ClickHouse Bronze.
Se agregan columnas de auditoría: `_ingested_at` y `_source`.
Las fechas nulas (NaT) se convierten a None para compatibilidad con ClickHouse.

In [3]:
def cargar_a_bronze(nombre_tabla, query, exclude_cols=[]):
    print(f'Leyendo {nombre_tabla} desde MySQL...')
    df = pd.read_sql(query, PG_ENGINE)

    # Eliminar columnas incompatibles (BLOB, GEOMETRY, etc.)
    if exclude_cols:
        df = df.drop(columns=exclude_cols, errors='ignore')

    # Convertir NaT → None en columnas datetime (compatibilidad ClickHouse)
    for col in df.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns:
        df[col] = df[col].astype(object).where(df[col].notna(), None)

    # Convertir NaT → None en columnas object con fechas
    for col in df.columns:
        if str(df[col].dtype) == 'object':
            try:
                mask = df[col].isna()
                df[col] = df[col].astype(object)
                df.loc[mask, col] = None
            except:
                pass

    # Agregar columnas de auditoría
    df['_ingested_at'] = datetime.now()
    df['_source']      = 'mysql'
    df = df.where(pd.notnull(df), None)

    CH.execute(f'TRUNCATE TABLE {BRONZE_DB}.raw_{nombre_tabla}')
    CH.execute(f'INSERT INTO {BRONZE_DB}.raw_{nombre_tabla} VALUES', df.to_dict('records'))
    print(f'  ✓ {len(df):,} filas cargadas en bronze.raw_{nombre_tabla}')

In [4]:
# Definición de tablas con SELECT explícito por tabla
# Se excluye la columna 'picture' de staff (tipo BLOB incompatible)
tablas = {
    'actor':         ('SELECT actor_id, first_name, last_name, last_update FROM actor', []),
    'address':       ('SELECT address_id, address, address2, district, city_id, postal_code, phone, last_update FROM address', []),
    'category':      ('SELECT category_id, name, last_update FROM category', []),
    'city':          ('SELECT city_id, city, country_id, last_update FROM city', []),
    'country':       ('SELECT country_id, country, last_update FROM country', []),
    'customer':      ('SELECT customer_id, store_id, first_name, last_name, email, address_id, active, create_date, last_update FROM customer', []),
    'film':          ('SELECT film_id, title, description, release_year, language_id, original_language_id, rental_duration, rental_rate, length, replacement_cost, rating, special_features, last_update FROM film', []),
    'film_actor':    ('SELECT actor_id, film_id, last_update FROM film_actor', []),
    'film_category': ('SELECT film_id, category_id, last_update FROM film_category', []),
    'inventory':     ('SELECT inventory_id, film_id, store_id, last_update FROM inventory', []),
    'language':      ('SELECT language_id, name, last_update FROM language', []),
    'payment':       ('SELECT payment_id, customer_id, staff_id, rental_id, amount, payment_date, last_update FROM payment', []),
    'rental':        ('SELECT rental_id, rental_date, inventory_id, customer_id, return_date, staff_id, last_update FROM rental', []),
    'staff':         ('SELECT staff_id, first_name, last_name, address_id, email, store_id, active, username, last_update FROM staff', ['picture']),
    'store':         ('SELECT store_id, manager_staff_id, address_id, last_update FROM store', []),
}

In [5]:
# Ejecutar carga Bronze — todas las tablas
print('=== ETL MySQL → Bronze ===')
for tabla, (query, exclude) in tablas.items():
    cargar_a_bronze(tabla, query, exclude)
print('\n=== Carga Bronze completada ===')

=== ETL MySQL → Bronze ===
Leyendo actor desde MySQL...
  ✓ 200 filas cargadas en bronze.raw_actor
Leyendo address desde MySQL...
  ✓ 603 filas cargadas en bronze.raw_address
Leyendo category desde MySQL...
  ✓ 16 filas cargadas en bronze.raw_category
Leyendo city desde MySQL...
  ✓ 600 filas cargadas en bronze.raw_city
Leyendo country desde MySQL...
  ✓ 109 filas cargadas en bronze.raw_country
Leyendo customer desde MySQL...
  ✓ 599 filas cargadas en bronze.raw_customer
Leyendo film desde MySQL...
  ✓ 1,000 filas cargadas en bronze.raw_film
Leyendo film_actor desde MySQL...
  ✓ 5,462 filas cargadas en bronze.raw_film_actor
Leyendo film_category desde MySQL...
  ✓ 1,000 filas cargadas en bronze.raw_film_category
Leyendo inventory desde MySQL...
  ✓ 4,581 filas cargadas en bronze.raw_inventory
Leyendo language desde MySQL...
  ✓ 6 filas cargadas en bronze.raw_language
Leyendo payment desde MySQL...
  ✓ 16,044 filas cargadas en bronze.raw_payment
Leyendo rental desde MySQL...
  ✓ 16,044 

In [6]:
# Verificación de conteos clave
print('=== VERIFICACIÓN BRONZE ===')
checks = [
    ('rental',   16044),
    ('film',     1000),
    ('customer', 599),
    ('payment',  16044),
    ('address',  603),
    ('actor',    200),
]
for tabla, esperado in checks:
    result = CH.execute(f'SELECT count() FROM bronze.raw_{tabla}')[0][0]
    status = '✓' if result == esperado else '✗ ALERTA'
    print(f'  {status} raw_{tabla}: {result:,} filas (esperado: {esperado:,})')

=== VERIFICACIÓN BRONZE ===
  ✓ raw_rental: 16,044 filas (esperado: 16,044)
  ✓ raw_film: 1,000 filas (esperado: 1,000)
  ✓ raw_customer: 599 filas (esperado: 599)
  ✓ raw_payment: 16,044 filas (esperado: 16,044)
  ✓ raw_address: 603 filas (esperado: 603)
  ✓ raw_actor: 200 filas (esperado: 200)
